In [86]:
import pandas as pd

df1=pd.DataFrame()
df1=pd.read_excel('../data/1.xlsx')
df1=df1.dropna()

def angle_v2(vec1, vec2):
    #compute angle between 2 vectors. ... from google
    #import math
    import numpy as np

    np.array 
    
    # 각 벡터의 각도 계산 (x축 기준)
    angle1 = np.arctan2(vec1[:,1], vec1[:,0])
    angle2 = np.arctan2(vec2[:,1], vec2[:,0])

    # 두 각도의 차이 계산
    angle_diff_rad = angle2 - angle1

    # 절대값 및 360도 조정 (필요에 따라)
    angle_diff_deg = np.degrees(angle_diff_rad)
    #angle_diff_deg = np.where( angle_diff_deg < 0, angle_diff_deg + 360, angle_diff_deg )

    return angle_diff_deg


In [87]:
#df_ant - projection to z axis for anterior view
df_ant = df1[['l_hip_x','l_hip_y','l_knee_x','l_knee_y','l_ankle_x','l_ankle_y']]

pd.set_option('mode.chained_assignment', None)
df_ant['l_tibia_x'] = df_ant['l_knee_x'] - df_ant['l_ankle_x']
df_ant['l_tibia_y'] = df_ant['l_knee_y'] - df_ant['l_ankle_y']
df_ant['l_femur_x'] = df_ant['l_hip_x'] - df_ant['l_knee_x']
df_ant['l_femur_y'] = df_ant['l_hip_y'] - df_ant['l_knee_y']

#lateral view
df_lat = df1[['torso_y', 'torso_z', 'waist_y','waist_z','l_hip_y','l_hip_z','l_knee_y','l_knee_z','l_ankle_y','l_ankle_z']]

df_lat['tor_wai_y'] = df_lat['torso_y'] - df_lat['waist_y']
df_lat['tor_wai_z'] = df_lat['torso_z'] - df_lat['waist_z']
df_lat['l_hip_wai_y'] = df_lat['waist_y'] - df_lat['l_hip_y']
df_lat['l_hip_wai_z'] = df_lat['waist_z'] - df_lat['l_hip_z']
df_lat['l_tibia_y'] = df_lat['l_knee_y'] - df_lat['l_ankle_y']
df_lat['l_tibia_z'] = df_lat['l_knee_z'] - df_lat['l_ankle_z']


In [88]:
#knees move inward, outward   : anterior view
angle_v2(df_ant[['l_tibia_x','l_tibia_y']].values,       df_ant[['l_femur_x','l_femur_y']].values     )[0:30]

array([-1.04603804, -0.58893883, -0.58893883, -0.58893883,  6.05217421,
        6.05217421,  6.05217421,  7.00831698,  7.00831698,  7.00831698,
        7.00831698,  3.63826577,  5.04013666,  5.59643378,  4.93305672,
        5.04789869,  5.45474378,  6.47240246,  6.65108094,  6.65108094,
        6.65108094,  6.65108094,  5.73144701,  5.73144701,  5.73144701,
        4.13124741,  4.13124741,  4.13124741,  3.28872779,  3.28872779])

In [89]:
#low back arch   : lateral view
angle_v2(df_lat[['tor_wai_z', 'tor_wai_y']].values, df_lat[['l_hip_wai_z', 'l_hip_wai_y']].values)[1:45]

array([19.52826228, 19.52826228, 19.52826228, 20.47174008, 20.47174008,
       20.47174008, 23.73355201, 23.73355201, 23.73355201, 23.73355201,
       40.25215899, 36.58546662, 33.60983192, 49.37144954, 53.70712933,
       46.33269846, 47.13091661, 52.8805865 , 52.8805865 , 52.8805865 ,
       52.8805865 , 58.33710421, 58.33710421, 58.33710421, 35.0178035 ,
       35.0178035 , 35.0178035 , 16.99112836, 16.99112836, 14.70892681,
       14.70892681, 14.70892681, 17.2000072 , 17.2000072 , 17.2000072 ,
       20.71642556, 20.71642556, 20.71642556, 26.24926783, 26.24926783,
       26.19067844, 17.44323576, 11.04570688,  7.11688621])

In [90]:
#torso lean forward  : lateral view
angle_v2(df_lat[['l_tibia_z','l_tibia_y']].values, df_lat[['tor_wai_z','tor_wai_y']].values)[1:30]

array([-34.02183777, -34.02183777, -34.02183777,  -7.34679683,
        -7.34679683,  -7.34679683,  -8.39086991,  -8.39086991,
        -8.39086991,  -8.39086991, -39.14362165, -39.85861378,
       -42.28125645, -44.03910633, -45.81277677, -47.1871012 ,
       -48.74229991, -49.11906927, -49.11906927, -49.11906927,
       -49.11906927, -46.83035541, -46.83035541, -46.83035541,
       -43.27326755, -43.27326755, -43.27326755, -42.39249222,
       -42.39249222])

In [98]:
# do noise reduction and 
# find local minima of movement of body ( minimum position of torso along y axis during the assessment)
import numpy as np
from scipy.signal import argrelextrema
import scipy

In [92]:
x = df1['torso_y'].rolling(window=25).mean().values
minima_indices = argrelextrema(x, np.less)


In [93]:
m = minima_indices[0]
m

array([ 60, 166, 255, 342, 427])

In [94]:
#knees move inward, outward   : anterior view
#+ means outward (o형),  - means inward (x형 )
angle_v2(df_ant[['l_tibia_x','l_tibia_y']].loc[m,:].values,       df_ant[['l_femur_x','l_femur_y']].loc[m,:].values     )

array([-3.43619688,  6.19388807,  0.1199091 , -6.36560583,  3.52982533])

In [95]:
#low back arch
# "large + " value means Excessive low back arch
angle_v2(df_lat[['tor_wai_z', 'tor_wai_y']].loc[m,:].values, df_lat[['l_hip_wai_z','l_hip_wai_y']].loc[m,:].values)

array([ 4.25279603, 18.80122793,  9.82376877, 11.67896596,  9.13214196])

In [96]:
#torso lean forward  : lateral view
# large plus value means excessive forward lean

angle_v2(df_lat[['l_tibia_z','l_tibia_y']].loc[m,:].values, df_lat[['tor_wai_z','tor_wai_y']].loc[m,:].values)

array([-37.42534863, -36.91063349, -36.05264975, -47.88251918,
        -2.85164222])

In [97]:
#torso lean forward  : lateral view

angle_v2(df_lat[['l_tibia_z','l_tibia_y']].values, df_lat[['tor_wai_z','tor_wai_y']].values)[30:50]

array([-43.77418098, -43.77418098, -43.77418098, -44.22042762,
       -44.22042762, -44.22042762, -44.62393332, -44.62393332,
       -44.62393332, -43.88401119, -43.88401119, -41.89189449,
       -41.84691539, -41.4524965 , -40.05328272, -38.95191127,
       -38.36155005, -38.12047115, -38.12047115, -38.12047115])

In [61]:
m

array([ 50, 137, 212, 283, 354])